# EPANET Parser: Ruleset Listing and Validation Example

This notebook demonstrates how to:
1. List available rulesets (core and custom)
2. Parse and validate an EPANET network model
3. Inspect validation results
4. Use custom rulesets (e.g., MILP ruleset)
5. Export validation results as JSON

In [ ]:
# Setup: find test data files relative to this notebook
import os
from pathlib import Path

# Get the directory containing this notebook
notebook_dir = Path(os.getcwd())
# Go up one level to project root, then into tests/data
project_root = notebook_dir.parent
test_data_dir = project_root / "tests" / "data"

valid_network_path = test_data_dir / "valid_network.json"
invalid_network_path = test_data_dir / "invalid_network.json"

print(f"Notebook dir: {notebook_dir}")
print(f"Project root: {project_root}")
print(f"Test data dir: {test_data_dir}")
print(f"Valid network exists: {valid_network_path.exists()}")
print(f"Invalid network exists: {invalid_network_path.exists()}")

In [ ]:
# Import required modules
from epanetparser.core.validation import (
    list_rulesets,
    validate,
    ValidationContext,
    Severity
)
from epanetparser.core.epanettypes.network import WNTREPANETNetwork
import json

## 1. List Available Rulesets

The `list_rulesets()` function returns structured information about all discovered rulesets, including both core and custom rulesets.

In [ ]:
# List all available rulesets
rulesets = list_rulesets()

print(f"Found {len(rulesets)} ruleset(s):\n")
for rs in rulesets:
    kind = "core" if rs["is_core"] else "custom"
    print(f"  {rs['key']} ({kind})")
    print(f"    Name: {rs['name']}")
    print(f"    Version: {rs['version']}")
    print(f"    Description: {rs['description']}")
    print(f"    Total rules: {rs['rule_count']}")
    print(f"    Component rules: {len(rs['component_rules'])}")
    print(f"    Network rules: {len(rs['network_rules'])}")
    print()

## 2. Parse a Network Model

Parse an EPANET model from a JSON file (WNTR format).

In [ ]:
# Load a network from JSON file
network = WNTREPANETNetwork.from_file(str(valid_network_path))[0]

# Show basic network info
print(f"Network: {network.name}")
print(f"Nodes: {len(network.nodes)}")
print(f"Links: {len(network.links)}")
print(f"Curves: {len(network.curves)}")
print(f"Patterns: {len(network.patterns)}")

## 3. Validate with Core Ruleset (Default)

Validate the network using the default core ruleset. The core ruleset checks simulator-agnostic EPANET requirements.

In [ ]:
# Validate with core ruleset only (default)
report = network.validate()

print(f"Valid: {report.is_valid}")
print(f"Total issues: {len(report)}")
print(f"Errors: {len(report.errors)}")
print(f"Warnings: {len(report.warnings)}")
print(f"Info: {len(report.info)}")

# Show all issues
for issue in report:
    print(f"  [{issue.severity}] {issue.component_name or issue.component_type}: {issue.code} - {issue.message}")

## 4. Validate with Custom Ruleset (MILP)

Validate the same network with both the core ruleset and the MILP custom ruleset. The MILP ruleset encodes constraints for a specific Mixed Integer Linear Programming pump scheduling tool.

In [ ]:
# Validate with core + MILP custom ruleset
report_milp = network.validate(["epanet_core", "milp"])

print(f"Valid: {report_milp.is_valid}")
print(f"Total issues: {len(report_milp)}")
print(f"Errors: {len(report_milp.errors)}")
print(f"Warnings: {len(report_milp.warnings)}")

# Show MILP-specific issues
milp_issues = [i for i in report_milp if i.ruleset_key == "milp"]
print(f"\nMILP ruleset issues: {len(milp_issues)}")
for issue in milp_issues:
    print(f"  [{issue.severity}] {issue.component_name}: {issue.code} - {issue.message}")

## 5. Inspect Validation Results in Detail

The validation report provides rich query capabilities to inspect issues by severity, code, rule, or component.

In [ ]:
# Load an invalid network to see issues
invalid_network = WNTREPANETNetwork.from_file(str(invalid_network_path))[0]
report_invalid = invalid_network.validate(["epanet_core", "milp"])

print(f"Valid: {report_invalid.is_valid}")
print(f"Total issues: {len(report_invalid)}")

# Group by severity
print("\n=== By Severity ===")
for severity in Severity:
    issues = report_invalid.by_severity(severity)
    print(f"  {severity.name}: {len(issues)}")

# Group by component
print("\n=== By Component ===")
grouped = report_invalid.grouped_by_component()
for component, issues in grouped.items():
    print(f"  {component}: {len(issues)} issues")
    for issue in issues[:3]:  # Show first 3
        print(f"    [{issue.severity}] {issue.code}: {issue.message}")
    if len(issues) > 3:
        print(f"    ... and {len(issues) - 3} more")

## 6. Access Component Data in Issues

Each ValidationIssue now includes the full `component_data` dictionary, allowing self-contained reporting without needing the original model.

In [ ]:
# Show component_data for issues
for issue in report_invalid:
    if issue.component_data:
        print(f"\nIssue: {issue.code} ({issue.component_name})")
        print(f"  Component type: {issue.component_type}")
        print(f"  Component data keys: {list(issue.component_data.keys())}")
        # Show a few key fields
        for key in ["name", "type", "diameter", "elevation", "pump_curve_name"]:
            if key in issue.component_data:
                print(f"    {key}: {issue.component_data[key]}")

## 7. Use RuleViolation for Structured Output

Rules can raise `RuleViolation` with structured data including failing fields and full component data.

In [ ]:
from epanetparser.core.validation import RuleViolation, rule, Severity
from epanetparser.core.epanettypes import WNTREPANETLink
from types import ModuleType

# Create a test module with a rule using RuleViolation
test_module = ModuleType('test_custom_rule')

@rule("WNTREPANETLink")
def rule_pump_must_have_curve(link):
    """A pump must reference a valid pump curve."""
    curve = link.data.get("pump_curve_name")
    if curve:
        # In a real rule, you'd check against network.index.curves
        raise RuleViolation(
            f"Pump references curve '{curve}' which does not exist",
            failing_fields=["pump_curve_name"],
            component_data=link.data,
            curve=curve,
            pump=link.name
        )

rule_pump_must_have_curve.__module__ = "test_custom_rule"
test_module.rule_pump_must_have_curve = rule_pump_must_have_curve

# Collect and inspect the rule
from epanetparser.core.validation import collect_rules
comp_rules, _ = collect_rules(test_module)
print(f"Rule: {comp_rules[0].rule_id}")
print(f"Code: {comp_rules[0].code}")
print(f"Severity: {comp_rules[0].severity}")

## 8. Export Results as JSON

The validation report can be serialized to JSON for programmatic processing or storage.

In [ ]:
# Export report as JSON
json_output = report_invalid.as_dict()

# Pretty print first few issues
print(json.dumps({
    "is_valid": json_output["is_valid"],
    "counts": json_output["counts"],
    "issues": json_output["issues"][:2]  # First 2 issues
}, indent=2))

## 9. Using ValidationContext for Explicit Control

For more explicit control over ruleset selection, use `ValidationContext` directly.

In [ ]:
# Explicit validation context
context = ValidationContext(core="epanet_core", custom=["milp"])
report = validate(network, context)

print(f"Rulesets used: {context.ruleset_keys}")
print(f"Valid: {report.is_valid}")

# You can also list rulesets from the context's registry
registry = context.registry
print(f"\nAvailable core rulesets: {registry.core_keys()}")
print(f"Available custom rulesets: {registry.custom_keys()}")

# Get details of a specific ruleset
core_rs = registry.get("epanet_core")
print(f"\nCore ruleset: {core_rs.name} v{core_rs.version}")
print(f"Component rules: {len(core_rs.component_rules)}")
print(f"Network rules: {len(core_rs.network_rules)}")